In [1]:
from scipy.integrate import odeint
import numpy as np
import scipy.constants as const
from matplotlib import pyplot as plt
from scipy.interpolate import interp1d
from scipy.integrate import quad
import collections

In [ ]:
Ic = 1e-6
Rsg = 10e3
Rn = Rsg
Cj = 2e-15
Rin = 100e0
Vgap = 4*Ic*Rn/np.pi #Gap voltage linked to IcRn, SIS relation
Vn = 1.1 #0.1*Vgap/Ic/Rin
V0 = Vn*Rin*Ic
phi0 = const.value('mag. flux quantum')
wp = 1/np.sqrt(Cj*phi0/2/np.pi/Ic)

params = {'Vn':Vn, 'Rsg': Rsg, 'Rn':Rn, 'Vgap':Vgap, 'Cj':Cj, 'Rin':Rin, 'wp':wp}
print(Vn)

In [ ]:
taustop = 20* wp * Rsg*Cj
params['taustop'] = taustop
print(params)

In [4]:
def Rj(V,params):
    Vgap = params['Vgap']
    if V>Vgap:
        return params['Rn']
    elif V<=Vgap:
        return params['Rsg']

def Q(V,params):
    Cj = params['Cj']
    Rin = params['Rin']
    wp = params['wp']
    return Cj*wp/(1/Rj(V,params)+1/Rin)

def Vs(t,params):
    if not hasattr(t,'__len__'):
        t = np.array([t])
    result = []
    for x in t:
        if x <params['taustop']/2:
            result.append(x/(params['taustop']/2)*params['Vn'])
        else:
            result.append(params['Vn'])
    #return params['Vn']
    result = np.ones(len(t))*params['Vn']
    return np.array(result)

def VJJmodel(z,t,params):
    f = z[0]
    g = z[1]
    V = phi0/2/np.pi*g
    fp = g
    gp = Vs(t,params)-(1/Q(V,params)*g+np.sin(f))
    return np.array([fp,gp])

In [ ]:
ts = np.linspace(0,2*taustop,2000)
sols = odeint(VJJmodel,[0,0],ts, args=tuple([params]) )
solsfunc = interp1d(ts,sols.T)
Vjunc = lambda x: phi0/2/np.pi * wp * solsfunc(x)[1]
taurep = 2*np.pi/np.average(np.diff(sols[:,0])/np.diff(ts))

In [ ]:
fig = plt.figure(figsize=(10,10))
plt.clf()
plt.subplot(3, 1, 1)
plt.title('Voltage/Vgap')
plt.plot(ts,Vjunc(ts) / Vgap)
plt.subplot(3, 1, 2)
plt.title('Current')
plt.plot(ts,np.sin(solsfunc(ts)[0]))
plt.plot(ts,(Vs(ts,params)*Ic*Rin-Vjunc(ts))/Rin/Ic)
plt.ylim([-2,2])
plt.subplot(3, 1, 3)
plt.title('Phase')
plt.plot(ts,solsfunc(ts)[0])
plt.show()
plt.close(fig)

In [8]:
import numpy as np
from scipy.integrate import odeint
from scipy.interpolate import interp1d

# 假設這是你的 VJJmodel 函數
def VJJmodel(z, t, params):
    f, g = z
    V, Q, Vs, phi0, wp = params
    fp = g
    gp = Vs(t, params) - (1/Q(V, params) * g + np.sin(f))
    return np.array([fp, gp])

# 假設這是你的 Vs 函數
def Vs(t, params):
    # 確保返回的是標量值
    return np.sin(t)

# 假設這是你的 Q 函數
def Q(V, params):
    # 確保返回的是標量值
    return 1.0

# 假設這是你的參數和其他變數
phi0 = 1.0
wp = 1.0
params = (1.0, Q, Vs, phi0, wp)
taustop = 10  # 假設這是 taustop 的值
ts = np.linspace(0, 2 * taustop, 2000)

# 使用 odeint 進行積分
sols = odeint(VJJmodel, [0, 0], ts, args=(params,))
solsfunc = interp1d(ts, sols.T)
Vjunc = lambda x: phi0 / 2 / np.pi * wp * solsfunc(x)[1]

In [ ]:
pip install stlab

In [ ]:
import numpy as np

from scipy.signal import argrelextrema
import scipy.integrate as integrate
import scipy.special as special
from scipy.integrate import odeint
from scipy.constants import constants as const
from scipy.fftpack import fft, fftfreq

import matplotlib.pyplot as plt

from rcsj_iv import *
from funcs import ensure_dir

##################
##################

currents = np.arange(0.,2.01,0.01)
all_currents = np.concatenate([currents[:-1],currents[::-1]])
print(all_currents)
time = np.arange(0,100,0.01)

betas = np.logspace(-2,2,41)

iv = [rcsj_iv(all_currents,beta=bb) for bb in betas]


# Plotting
[plt.plot(ivv[0],ivv[1]/bb,'.-',label=str(bb)) for ivv,bb in zip(iv,betas)]
plt.xlabel(r'$I/I_c$')
plt.ylabel(r'$V/\beta_c$')
plt.legend()
plt.savefig('../plots/ivcs_updown.png',bbox_to_inches='tight')
plt.show()
plt.close()


In [ ]:
currents = np.arange(0.,2.01,0.01)
all_currents = np.concatenate([currents[:-1],currents[::-1]])
print(all_currents)
time = np.arange(0,500,0.01)

betas = np.logspace(-2,2,41)

iv = [rcsj_iv(all_currents, damping=('beta', bb)) for bb in betas]

# Plotting
[plt.plot(ivv[0],ivv[1]/bb,'.-',label=str(bb)) for ivv,bb in zip(iv,betas)]
plt.xlabel(r'$I/I_c$')
plt.ylabel(r'$V/\beta_c$')
plt.legend()
plt.savefig('../plots/ivcs_updown.png',bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
currents = np.arange(0.,2.01,0.01)
all_currents = np.concatenate([currents[:-1],currents[::-1]])
print(all_currents)
time = np.arange(0,10,0.01)

betas = np.logspace(-1,1,11)

iv = [rcsj_iv(all_currents, damping=('beta', bb)) for bb in betas]

# Plotting
[plt.plot(ivv[0],ivv[1]/bb,'.-',label=str(bb)) for ivv,bb in zip(iv,betas)]
plt.xlabel(r'$I/I_c$')
plt.ylabel(r'$V/\beta_c$')
plt.legend()
plt.savefig('../plots/ivcs_updown.png',bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
currents = np.arange(0.,2.01,0.01)
all_currents = np.concatenate([currents[:-1],currents[::-1]])
print(all_currents)
time = np.arange(0,1,0.01)

betas = np.logspace(-1,1,11)

iv = [rcsj_iv(all_currents, damping=('beta', bb)) for bb in betas]

# Plotting
[plt.plot(ivv[0],ivv[1]/bb,'.-',label=str(bb)) for ivv,bb in zip(iv,betas)]
plt.xlabel(r'$I/I_c$')
plt.ylabel(r'$V/\beta_c$')
plt.legend()
plt.savefig('../plots/ivcs_updown.png',bbox_inches='tight')
plt.show()
plt.close()

In [ ]:
import jax
import jax.numpy as jnp
import diffrax
from jax import config
config.update("jax_enable_x64", True)
import matplotlib.pyplot as plt

# Parameters
Ic = jnp.float64(1e-6)  # Critical current (A)
R = jnp.float64(1)  # Shunt resistance (Ohm)
C = jnp.float64(1e-6)  # Shunt capacitance (F)
phi0 = jnp.float64(2.067e-15)  # Magnetic flux quantum (Wb)
hbar_over_2e = phi0 / (2 * jnp.pi)  # Reduced flux quantum
t = jnp.linspace(0, 10e-8, 5000)  # Time array

# Define the RCSJ differential equation using JAX
def rcsj_ode(t, y, args):
    phi, phi_t = y
    I_ext = args["I_ext"]
    V = hbar_over_2e * phi_t
    dphi_dt = phi_t
    dphi_t_dt = (I_ext - Ic * jnp.sin(phi) - V / R) / (C * hbar_over_2e)
    return jnp.array([dphi_dt, dphi_t_dt])

# Simulate the RCSJ IV curve by sweeping current from -2 µA to 2 µA
currents_applied = jnp.linspace(-5e-6, 5e-6, 501)
voltages_measured = []

# Solve the RCSJ equation for each applied current
for I_ext in currents_applied:
    y0 = jnp.array([0.0, 0.0])
    solver = diffrax.Tsit5()
    term = diffrax.ODETerm(rcsj_ode)
    sol = diffrax.diffeqsolve(
        term,
        solver,
        t0=t[0],
        t1=t[-1],
        dt0=1e-11,
        y0=y0,
        args={"I_ext": I_ext},
        saveat=diffrax.SaveAt(ts=t),
        max_steps=10000000
    )
    phi_t = sol.ys[:, 1]
    V_junction = hbar_over_2e * phi_t
    voltages_measured.append(jnp.mean(V_junction[-100:]))

voltages_measured = jnp.array(voltages_measured)

# Plot the RCSJ response for applied current from -2 µA to 2 µA
plt.figure(figsize=(8, 6))
plt.plot(currents_applied, voltages_measured, label='RCSJ IV Curve (JAX)', color='blue')
plt.xlabel('Applied Current (A)')
plt.ylabel('Voltage (V)')
plt.title('RCSJ IV Curve using JAX and Diffrax')
plt.grid(True)
plt.legend()
plt.show()


In [ ]:
import jax
import jax.numpy as jnp
import diffrax
from jax import config
config.update("jax_enable_x64", True)
import matplotlib.pyplot as plt

# Parameters
Ic = jnp.float64(1e-6)  # Critical current (A)
R = jnp.float64(1)  # Shunt resistance (Ohm)
C = jnp.float64(1e-6)  # Shunt capacitance (F)
phi0 = jnp.float64(2.067e-15)  # Magnetic flux quantum (Wb)
hbar_over_2e = phi0 / (2 * jnp.pi)  # Reduced flux quantum
t = jnp.linspace(0, 2e-7, 500)  # Time array

# Define the RCSJ differential equation using JAX
def rcsj_ode(t, y, args):
    phi, phi_t = y
    I_ext = args["I_ext"]
    V = hbar_over_2e * phi_t
    dphi_dt = phi_t
    dphi_t_dt = (I_ext - Ic * jnp.sin(phi) - V / R) / (C * hbar_over_2e)
    return jnp.array([dphi_dt, dphi_t_dt])

# Simulate the RCSJ IV curve by sweeping current from -2 µA to 2 µA
currents_applied = jnp.linspace(-5e-6, 5e-6, 501)
voltages_measured = []

# Solve the RCSJ equation for each applied current
for I_ext in currents_applied:
    y0 = jnp.array([0.0, 0.0])
    solver = diffrax.Tsit5()
    term = diffrax.ODETerm(rcsj_ode)
    sol = diffrax.diffeqsolve(
        term,
        solver,
        t0=t[0],
        t1=t[-1],
        dt0=1e-11,
        y0=y0,
        args={"I_ext": I_ext},
        saveat=diffrax.SaveAt(ts=t),
        max_steps=10000000
    )
    phi_t = sol.ys[:, 1]
    V_junction = hbar_over_2e * phi_t
    voltages_measured.append(jnp.mean(V_junction[-100:]))

voltages_measured = jnp.array(voltages_measured)

# Plot the RCSJ response for applied current from -2 µA to 2 µA
plt.figure(figsize=(8, 6))
plt.plot(currents_applied, voltages_measured, label='RCSJ IV Curve (JAX)', color='blue')
plt.xlabel('Applied Current (A)')
plt.ylabel('Voltage (V)')
plt.title('RCSJ IV Curve using JAX and Diffrax')
plt.grid(True)
plt.legend()
plt.show()


In [ ]:
import jax
import jax.numpy as jnp
import diffrax
from jax import config
config.update("jax_enable_x64", True)
import plotly.graph_objects as go

# Parameters
Ic = jnp.float64(1e-6)  # Critical current (A)
R = jnp.float64(1)  # Shunt resistance (Ohm)
C = jnp.float64(1e-6)  # Shunt capacitance (F)
phi0 = jnp.float64(2.067e-15)  # Magnetic flux quantum (Wb)
hbar_over_2e = phi0 / (2 * jnp.pi)  # Reduced flux quantum
t = jnp.linspace(0, 2e-7, 500)  # Time array

# Define the RCSJ differential equation using JAX
def rcsj_ode(t, y, args):
    phi, phi_t = y
    I_ext = args["I_ext"]
    V = hbar_over_2e * phi_t
    dphi_dt = phi_t
    dphi_t_dt = (I_ext - Ic * jnp.sin(phi) - V / R) / (C * hbar_over_2e)
    return jnp.array([dphi_dt, dphi_t_dt])

# Simulate the RCSJ IV curve by sweeping current from -2 µA to 2 µA
currents_applied = jnp.linspace(-5e-6, 5e-6, 501)
voltages_measured = []

# Solve the RCSJ equation for each applied current
for I_ext in currents_applied:
    y0 = jnp.array([0.0, 0.0])
    solver = diffrax.Tsit5()
    term = diffrax.ODETerm(rcsj_ode)
    sol = diffrax.diffeqsolve(
        term,
        solver,
        t0=t[0],
        t1=t[-1],
        dt0=1e-11,
        y0=y0,
        args={"I_ext": I_ext},
        saveat=diffrax.SaveAt(ts=t),
        max_steps=10000000
    )
    phi_t = sol.ys[:, 1]
    V_junction = hbar_over_2e * phi_t
    voltages_measured.append(jnp.mean(V_junction[-100:]))

voltages_measured = jnp.array(voltages_measured)

# Plot the RCSJ response for applied current from -2 µA to 2 µA using Plotly
fig = go.Figure()
fig.add_trace(go.Scatter(x=currents_applied, y=voltages_measured, mode='lines', name='RCSJ IV Curve (JAX)', line=dict(color='blue')))
fig.update_layout(
    title='RCSJ IV Curve using JAX and Diffrax',
    xaxis_title='Applied Current (A)',
    yaxis_title='Voltage (V)',
    legend=dict(x=0, y=1),
    margin=dict(l=40, r=40, t=40, b=40),
    template='plotly_white'
)
fig.show()

In [ ]:
import jax
import jax.numpy as jnp
import diffrax
from jax import config
config.update("jax_enable_x64", True)
import plotly.graph_objects as go

# Parameters
Ic = jnp.float64(1e-6)  # Critical current (A)
R = jnp.float64(1)  # Shunt resistance (Ohm)
C = jnp.float64(1e-6)  # Shunt capacitance (F)
phi0 = jnp.float64(2.067e-15)  # Magnetic flux quantum (Wb)
hbar_over_2e = phi0 / (2 * jnp.pi)  # Reduced flux quantum
t = jnp.linspace(0, 2e-7, 500)  # Time array

# Define the RCSJ differential equation using JAX
def rcsj_ode(t, y, args):
    phi, phi_t = y
    I_ext = args["I_ext"]
    V = hbar_over_2e * phi_t
    dphi_dt = phi_t
    dphi_t_dt = (I_ext - Ic * jnp.sin(phi) - V / R) / (C * hbar_over_2e)
    return jnp.array([dphi_dt, dphi_t_dt])

# Simulate the RCSJ IV curve by sweeping current from -2 µA to 2 µA
currents_applied = jnp.linspace(-5e-6, 5e-6, 501)
voltages_measured = []

# Solve the RCSJ equation for each applied current
for I_ext in currents_applied:
    y0 = jnp.array([0.0, 0.0])
    solver = diffrax.Tsit5()
    term = diffrax.ODETerm(rcsj_ode)
    sol = diffrax.diffeqsolve(
        term,
        solver,
        t0=t[0],
        t1=t[-1],
        dt0=1e-11,
        y0=y0,
        args={"I_ext": I_ext},
        saveat=diffrax.SaveAt(ts=t),
        max_steps=10000000
    )
    phi_t = sol.ys[:, 1]
    V_junction = hbar_over_2e * phi_t
    voltages_measured.append(jnp.mean(V_junction[-100:]))

voltages_measured = jnp.array(voltages_measured)

# Plot the RCSJ response for applied current from -2 µA to 2 µA using Plotly
fig = go.Figure()
fig.add_trace(go.Scatter(x=currents_applied, y=voltages_measured, mode='lines', name='RCSJ IV Curve (JAX)', line=dict(color='blue')))
fig.update_layout(
    title=f'RCSJ IV Curve<br>Ic={Ic} A, R={R} Ohm, C={C} F',
    xaxis_title='Applied Current (A)',
    yaxis_title='Voltage (V)',
    legend=dict(x=0, y=1),
    margin=dict(l=40, r=40, t=40, b=40),
    template='plotly_white'
)
fig.show()

# RCSJ IV Curve simulation

In [ ]:
import jax
import jax.numpy as jnp
import diffrax
from jax import config
config.update("jax_enable_x64", True)
import plotly.graph_objects as go

# Parameters
Ic = jnp.float64(1e-6)  # Critical current (A)
R = jnp.float64(1)  # Shunt resistance (Ohm)
C = jnp.float64(1e-6)  # Shunt capacitance (F)
phi0 = jnp.float64(2.067e-15)  # Magnetic flux quantum (Wb)
hbar_over_2e = phi0 / (2 * jnp.pi)  # Reduced flux quantum
t = jnp.linspace(0, 2e-7, 101)  # Time array

# Define the RCSJ differential equation using JAX
def rcsj_ode(t, y, args):
    phi, phi_t = y
    I_ext = args["I_ext"]
    V = hbar_over_2e * phi_t
    dphi_dt = phi_t
    dphi_t_dt = (I_ext - Ic * jnp.sin(phi) - V / R) / (C * hbar_over_2e)
    return jnp.array([dphi_dt, dphi_t_dt])

# Simulate the RCSJ IV curve by sweeping current from -2 µA to 2 µA
currents_applied = jnp.linspace(-2e-6, 2e-6, 101)
voltages_measured = []

# Solve the RCSJ equation for each applied current
for I_ext in currents_applied:
    y0 = jnp.array([0.0, 0.0])
    solver = diffrax.Tsit5()
    term = diffrax.ODETerm(rcsj_ode)
    sol = diffrax.diffeqsolve(
        term,
        solver,
        t0=t[0],
        t1=t[-1],
        dt0=1e-9,
        y0=y0,
        args={"I_ext": I_ext},
        saveat=diffrax.SaveAt(ts=t),
        max_steps=100000000
    )
    phi_t = sol.ys[:, 1]
    V_junction = hbar_over_2e * phi_t
    voltages_measured.append(jnp.mean(V_junction[-100:]))

voltages_measured = jnp.array(voltages_measured)

# Plot the RCSJ response for applied current from -2 µA to 2 µA using Plotly
fig = go.Figure()
fig.add_trace(go.Scatter(x=currents_applied, y=voltages_measured, mode='lines', name='RCSJ IV Curve', line=dict(color='blue')))
fig.update_layout(
    title=f'RCSJ IV Curve<br>Ic={Ic} A, R={R} Ohm, C={C} F',
    xaxis_title='Applied Current (A)',
    yaxis_title='Voltage (V)',
    legend=dict(x=0, y=1),
    margin=dict(l=40, r=40, t=40, b=40),
    template='plotly_white'
)
fig.show()

# Function define

In [5]:
from scipy import stats
import numpy as np
import jax.numpy as jnp
import pint

# Initialize pint unit registry
ureg = pint.UnitRegistry()
ureg.formatter.default_format = '~P'  # Use SI prefix formatting

class SI:
    """Class containing SI units for easy access and formatting."""
    A = ureg.ampere
    V = ureg.volt
    Ω = ureg.ohm
    F = ureg.farad
    H = ureg.henry
    W = ureg.watt
    J = ureg.joule
    s = ureg.second
    m = ureg.meter
    g = ureg.gram
    C = ureg.coulomb
    K = ureg.kelvin
    dB = ureg.decibel

    @staticmethod
    def f(value, unit):
        """Format a value or array with its unit using SI prefixes."""
        # Handle array formatting: convert each element separately
        if isinstance(value, (jnp.ndarray, np.ndarray)):
            if value.ndim == 0:
                quantity = float(value) * unit
                return f"{quantity.to_compact():.2f~P}"
            else:
                formatted_values = [f"{(float(v) * unit).to_compact():.2f~P}" for v in value]
                return ", ".join(formatted_values)
        else:  
            quantity = float(value) * unit
            return f"{quantity.to_compact():.2f~P}"
       

class FitResult:
    def __init__(self, fit_name, slope, intercept, r_value, p_value, stderr, mean_value, step_size, noise_std, SNR):
        self.fit_name = fit_name
        self.slope = slope
        self.intercept = intercept
        self.r_value = r_value
        self.p_value = p_value
        self.stderr = stderr
        self.mean_value = mean_value
        self.step_size = step_size
        self.noise_std = noise_std
        self.SNR = SNR

def polyfit(fit_name, x, y):
    step_size = jnp.diff(x)[0].round(12)
    mean_value = jnp.mean(y)
    fit_result = stats.linregress(x, y)
    estimated_signal = fit_result.slope * x + fit_result.intercept
    noise = y - estimated_signal
    signal_power = jnp.mean(estimated_signal**2)
    noise_std = float(jnp.std(noise))
    SNR = 10 * jnp.log10(signal_power / noise_std**2)
    
    return FitResult(
        fit_name,
        fit_result.slope,
        fit_result.intercept,
        fit_result.rvalue,
        fit_result.pvalue,
        fit_result.stderr,
        mean_value,
        step_size,
        noise_std,
        SNR
    )

def constfit(fit_name, x, y):
    step_size = jnp.diff(x)[0].round(12)
    mean_value = jnp.mean(y)
    fit_result = jnp.polyfit(x, y, 0)
    slope = 0
    intercept = fit_result[0]
    estimated_signal = jnp.polyval(fit_result, x)
    noise = y - estimated_signal
    stderr = jnp.std(y, ddof=1) 
    signal_power = jnp.mean(estimated_signal**2)
    noise_std = float(jnp.std(noise))
    SNR = 10 * jnp.log10(signal_power / noise_std**2)

    return FitResult(
        fit_name,
        slope,
        intercept,
        0,
        1,
        stderr,
        mean_value,
        step_size,
        noise_std,
        SNR
    )

def print_fit_result(fit_result):
    print(f"{fit_result.fit_name}:")
    print(f"{'Step size:':>20} {SI.f(fit_result.step_size, SI.A)}")
    print(f"{'Mean:':>20} {fit_result.mean_value:.6f}")
    print(f"{'Slope:':>20} {fit_result.slope:.6f}")
    print(f"{'Intercept:':>20} {fit_result.intercept:.6f}")
    print(f"{'R²:':>20} {fit_result.r_value**2*100:.4f}%")
    print(f"{'P-value:':>20} {fit_result.p_value:.2e}")
    print(f"{'σ(SD):':>20} {fit_result.stderr:.2e}")
    print(f"{'Noise SD:':>20} {fit_result.noise_std:.2e}")
    print(f"{'SNR:':>20} {fit_result.SNR:.2f} dB\n")


In [ ]:
fit_result = jnp.polyfit(currents_applied, y, 0)

In [ ]:
R_fit = polyfit("R_fit", currents_applied, voltages_measured)
print_fit_result(R_fit)

Rn_fit = polyfit("Rn_fit", currents_applied[currents_applied > Ic], voltages_measured[currents_applied > Ic])
print_fit_result(Rn_fit)

dV_dI_fit = polyfit("dV_dI_fit", currents_applied, dV_dI)
print_fit_result(dV_dI_fit)

C_fit = constfit("C_fit", currents_applied, dV_dI)
print_fit_result(C_fit)

In [ ]:
import jax.numpy as jnp
import matplotlib.pyplot as plt
from scipy.stats import t

x = currents_applied
y = dV_dI

# Use jnp.polyfit to find the best-fit constant
params = jnp.polyfit(x, y, 0)
fitted_constant = params[0]

# Residuals and R-squared calculation
y_fitted = jnp.full_like(x, fitted_constant)
residuals = y - y_fitted
ss_res = jnp.sum(residuals**2)
ss_tot = jnp.sum((y - jnp.mean(y))**2)
r_squared = 1 - (ss_res / ss_tot)

# Standard deviation of the parameter estimate (for p-value)
stderr = jnp.std(residuals) / jnp.sqrt(len(x))

# Calculate the t-statistic and p-value for the constant parameter
t_stat = fitted_constant / stderr
p_value = 2 * (1 - t.cdf(jnp.abs(t_stat), df=len(x)-1))
mean = jnp.mean(y)

# Output results
print(f"Fitted constant value (intercept): {fitted_constant}")
print(f"Mean: {mean}")
print(f"Mean-constant: {mean-fitted_constant}")
print(f"Standard error of the fitted constant: {stderr}")
print(f"p-value: {p_value}")
print(f"R-squared: {r_squared}")

# Plot the data and the fitted constant line
plt.scatter(x, y, color='blue', label='Data')
plt.plot(x, y_fitted, color='red', label=f"Fitted constant y = {fitted_constant}")
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.show()

In [ ]:
import numpy as np
from scipy.stats import t

# Example data (y as the dependent variable with no x dependence)
y = dV_dI

# Sample mean (constant)
mean_y = np.mean(y)

# Standard deviation of y
std_y = np.std(y, ddof=1)  # ddof=1 for sample standard deviation

# Standard error of the mean
SE_y = std_y / np.sqrt(len(y))

# Degrees of freedom (n - 1 for constant model)
df = len(y) - 1
tinv = lambda p, df: abs(t.ppf(p / 2, df))

# Compute the critical t-value for 95% confidence level
alpha = 0.05
t_critical = tinv(alpha, df)

# Confidence interval for the constant
CI_constant = (mean_y - t_critical * SE_y, mean_y + t_critical * SE_y)


# Output results
print(f"Mean (constant value): {mean_y}")
print(f"Standard Deviation: {std_y}")
print(f"Standard Error of the Mean: {SE_y}")
print(f"critical t-value: {t_critical}")
print(f"Constant: {mean_y:.4f}±{t_critical * SE_y:.4f}")
print(f"95% Confidence Interval for constant: {CI_constant}")

In [ ]:
# Example data (x as independent variable and y as dependent variable)
x = currents_applied
y = np.array(voltages_measured)

# Add a constant to the independent variable (for intercept)
X = sm.add_constant(x)

# Fit the linear regression model
model = sm.OLS(y, X).fit()

# Get the slope (b1) and intercept (b0)
b0, b1 = model.params
# Standard errors for the slope and intercept
SE_b0, SE_b1 = model.bse

# Degrees of freedom (n - 2 for simple linear regression)
df = len(x) - 2

# Compute the critical t-value for 95% confidence level
alpha = 0.05
t_critical = tinv(alpha, df)

# Confidence intervals for the slope and intercept
CI_b0 = (b0 - t_critical * SE_b0, b0 + t_critical * SE_b0)
CI_b1 = (b1 - t_critical * SE_b1, b1 + t_critical * SE_b1)

# Output results
print(f"Slope (b1): {b1}")
print(f"Slope : {b1:.4f}±{t_critical * SE_b1:.4f}")
print(f"Intercept (b0): {b0}")
print(f"Intercept : {b0:.4f}±{t_critical * SE_b0:.4f}")

In [ ]:


# Two-sided inverse Students t-distribution
# p - probability, df - degrees of freedom
from scipy.stats import t
tinv = lambda p, df: abs(t.ppf(p/2, df))
ts = tinv(0.05, len(x)-2)
print(f"slope (95%): {R_fit.slope:.6f} +/- {ts*R_fit.stderr:.6f}")
print(f"intercept (95%): {R_fit.intercept:.6f}"
      f" +/- {ts*R_fit.stderr:.6f}")

In [ ]:
import jax
import jax.numpy as jnp
from scipy.stats import t

# Example data (y as the dependent variable with no x dependence)
y = dV_dI

# Sample mean (constant)
mean_y = jnp.mean(y)

# Standard deviation of y (using ddof=1 for sample std deviation)
std_y = jnp.std(y, ddof=1)

# Standard error of the mean
SE_y = std_y / jnp.sqrt(len(y))

# Degrees of freedom (n - 1 for constant model)
df = len(y) - 1

# Compute the critical t-value for 95% confidence level using scipy
alpha = 0.05
t_critical = tinv(alpha, df)

# Confidence interval for the constant
CI_constant = (mean_y - t_critical * SE_y, mean_y + t_critical * SE_y)

# Output results
print(f"Mean (constant value): {mean_y:.6f}")
print(f"95% Confidence Interval for constant: ({+-{t_critical * SE_y, mean_y})")

In [ ]:
import numpy as np
import statsmodels.api as sm
from scipy.stats import t

# Example data (x as independent variable and y as dependent variable)
x = currents_applied
y = voltages_measured

# Add a constant to the independent variable (for intercept)
X = sm.add_constant(x)

# Fit the linear regression model
model = sm.OLS(y, X).fit()

# Get the slope (b1) and intercept (b0)
b0, b1 = model.params
# Standard errors for the slope and intercept
SE_b0, SE_b1 = model.bse

# Degrees of freedom (n - 2 for simple linear regression)
df = len(x) - 1

# Compute the critical t-value for 95% confidence level
alpha = 0.05
t_critical = tinv(alpha, df)

# Confidence intervals for the slope and intercept
CI_b0 = (b0 - t_critical * SE_b0, b0 + t_critical * SE_b0)
CI_b1 = (b1 - t_critical * SE_b1, b1 + t_critical * SE_b1)

# Output results
print(f"Slope (b1): {b1}")
print(f"Intercept (b0): {b0}")
print(f"95% Confidence Interval for Slope: {CI_b1}")
print(f"95% Confidence Interval for Intercept: {CI_b0}")

In [ ]:
import numpy as np
import statsmodels.api as sm
from scipy.stats import t

# Example data (x as independent variable and y as dependent variable)
x = np.array([12, 15, 14, 10, 13, 16, 14, 17, 18, 11])
y = np.array([7, 10, 8, 5, 6, 12, 9, 11, 14, 6])

# Add a constant to the independent variable (for intercept)
X = sm.add_constant(x)

# Fit the linear regression model
model = sm.OLS(y, X).fit()

# Get the slope (b1) and intercept (b0)
b0, b1 = model.params
# Standard errors for the slope and intercept
SE_b0, SE_b1 = model.bse

# Degrees of freedom (n - 2 for simple linear regression)
df = len(x) - 2

# Compute the critical t-value for 95% confidence level
alpha = 0.05
t_critical = tinv(alpha, df)

# Confidence intervals for the slope and intercept
CI_b0 = (b0 - t_critical * SE_b0, b0 + t_critical * SE_b0)
CI_b1 = (b1 - t_critical * SE_b1, b1 + t_critical * SE_b1)

# Output results
print(f"Slope (b1): {b1}")
print(f"Intercept (b0): {b0}")
print(f"95% Confidence Interval for Slope: {CI_b1}")
print(f"95% Confidence Interval for Intercept: {CI_b0}")

# Testing

In [ ]:
%matplotlib ipympl
import os, sys
import time
import pyvisa
import warnings
import numpy as np
import qcodes as qc
import plotly.graph_objects as go
import matplotlib.pyplot as plt

def find_qcodes_local_dir():
    dirpath = os.getcwd()
    while True:
        dirpath, folder_name = os.path.split(dirpath)
        if folder_name == 'QCoDeS_local':
            return os.path.join(dirpath, folder_name)
        if not folder_name:  # Reached the root directory
            return None
qcodes_local_dir = find_qcodes_local_dir()
sys.path.append(f'{qcodes_local_dir}')

from tqdm import tqdm
from pprint import pprint
from time import sleep, monotonic, time
from IPython.display import clear_output
from qcodes.dataset import initialise_or_create_database_at
from qcodes.dataset.measurements import Measurement
from qcodes.utils.metadata import diff_param_values
from qcodes.dataset.plotting import plot_dataset, plot_by_id
from qcodes import Parameter, ManualParameter, ScaledParameter
from qcodes.instrument.specialized_parameters import ElapsedTimeParameter
# from sweeps_v2 import do1d, do2d, time_sweep, measure_until, do1d_until

print('Imported all modules, QCoDeS version:', qc.__version__, 'initialized')

# warnings.filterwarnings('ignore')
initialise_or_create_database_at(r"/Users/albert-mac/Library/CloudStorage/SynologyDrive-KeLab/09 Data/Fridge Data/2D material/PtTe2/PtTe2_NbTi_B7/PtTe2_NbTi_B7_2024-07-02_01.db")
qc.experiments()

In [ ]:
dataset = qc.load_by_id(80)
df = dataset.to_pandas_dataframe().reset_index()
para_list = dataset.parameters.split(",")

current = jnp.array(df[para_list[1]].tolist())
voltage = jnp.array(df[para_list[2]].tolist())
By = jnp.array(df[para_list[0]].tolist())
dV_dI = jnp.gradient(voltage, current)


# plot the heatmap X=By, Y=current, colorbar=dV_dI
fig = go.Figure()
fig.add_trace(go.Heatmap(
    z=dV_dI,
    x=By,
    y=current,
    colorscale='Viridis',
    colorbar=dict(title='dV/dI (V/A)')
))  
fig.show()
# plot the histogram of dV_dI
fig = go.Figure()
fig.add_trace(go.Histogram(x=dV_dI, histnorm='probability'))
fig.update_layout(
    title='Histogram of dV/dI',
    xaxis_title='dV/dI (V/A)',
    yaxis_title='Probability'
)
fig.show()

# plot x = By, y = dV_dI
fig = go.Figure()
fig.add_trace(go.Scatter(x=By, y=dV_dI, mode='lines', name='dV/dI'))
fig.update_layout(
    title='dV/dI vs By',
    xaxis_title='By (T)',
    yaxis_title='dV/dI (V/A)'
)
fig.show()

# FFT of By vs dV_dI
from scipy.fft import fft, fftfreq
# Number of sample points
N = len(By)
# sample spacing
T = 1.0 / 8000.0
x = np.linspace(0.0, N*T, N, endpoint=False)
yf = fft(By)
xf = fftfreq(N, T)[:N//2]
fig = go.Figure()
fig.add_trace(go.Scatter(x=xf, y=2.0/N * np.abs(yf[0:N//2]), mode='lines', name='FFT'))
fig.update_layout(
    title='FFT of By',
    xaxis_title='Frequency (Hz)',
    yaxis_title='Amplitude'
)
fig.show()


In [ ]:
import numpy as np
import jax.numpy as jnp
import plotly.graph_objects as go
from jax.numpy.fft import fft, fftfreq

dataset = qc.load_by_id(152)
df = dataset.to_pandas_dataframe().reset_index()
para_list = dataset.parameters.split(",")

current = jnp.array(df[para_list[1]].tolist())
voltage = jnp.array(df[para_list[2]].tolist())
By = jnp.array(df[para_list[0]].tolist())
dV_dI = jnp.gradient(voltage, current)

# plot the heatmap X=By, Y=current, colorbar=voltage
fig = go.Figure()
fig.add_trace(go.Heatmap(
    z=voltage,
    x=By,
    y=current,
    colorscale='Viridis',
    colorbar=dict(title='Voltage (V)')
))
fig.show()

# plot the heatmap X=By, Y=current, colorbar=dV_dI
fig = go.Figure()
fig.add_trace(go.Heatmap(
    z=dV_dI,
    x=By,
    y=current,
    colorscale='Viridis',
    colorbar=dict(title='dV/dI (V/A)')
))  
fig.show()

# plot the histogram of dV_dI
# Gaussian fit the tow peaks of dV_dI
from scipy.stats import norm
from scipy.optimize import curve_fit

def gaussian(x, mu, sigma, A):
    return A * np.exp(-0.5 * ((x - mu) / sigma)**2)

mu1, std1 = 0.5, 0.1
mu2, std2 = -0.5, 0.1
A1, A2 = 0.5, 0.5
popt1, _ = curve_fit(gaussian, dV_dI, norm.pdf(dV_dI, mu1, std1))
popt2, _ = curve_fit(gaussian, dV_dI, norm.pdf(dV_dI, mu2, std2))
mu1, std1, A1 = popt1
mu2, std2, A2 = popt2

fig = go.Figure()
fig.add_trace(go.Histogram(x=dV_dI, histnorm='probability'))
fig.add_trace(go.Scatter(x=dV_dI, y=norm.pdf(dV_dI, mu1, std1) * A1, mode='lines', name='Gaussian fit 1'))
fig.add_trace(go.Scatter(x=dV_dI, y=norm.pdf(dV_dI, mu2, std2) * A2, mode='lines', name='Gaussian fit 2'))
fig.update_layout(
    title='Histogram of dV/dI',
    xaxis_title='dV/dI (V/A)',
    yaxis_title='Probability'
)
fig.show()


# plot x = By, y = dV_dI
fig = go.Figure()
fig.add_trace(go.Scatter(x=By, y=dV_dI, mode='lines', name='dV/dI'))
fig.update_layout(
    title='dV/dI vs By',
    xaxis_title='By (T)',
    yaxis_title='dV/dI (V/A)'
)
fig.show()

# FFT of By vs dV_dI
# Number of sample points
N = len(By)
# sample spacing
T = 1 / 10
x = jnp.linspace(0.0, N*T, N, endpoint=False)
yf = fft(By)
xf = fftfreq(N, T)

# # Filter frequencies between 0 and 1 Hz
# mask = (xf >= 0) & (xf <= 1)
# xf = xf[mask]
# yf = yf[mask]

fig = go.Figure()
fig.add_trace(go.Scatter(x=xf, y=2.0/N * jnp.abs(yf), mode='lines', name='FFT'))
fig.update_layout(
    title='FFT of By (Frequencies between 0 and 1 Hz)',
    xaxis_title='Frequency (Hz)',
    yaxis_title='Amplitude'
)
fig.show()

In [ ]:
plot_by_id(80)

In [ ]:
from scipy import stats
from scipy.signal import find_peaks
import jax.numpy as jnp
import plotly.graph_objects as go
import pyperclip
from PIL import Image
import io
dataset = qc.load_by_id(135)

df = dataset.to_pandas_dataframe().reset_index()
para_list = dataset.parameters.split(",")

current = jnp.array(df[para_list[0]].tolist())
voltage = jnp.array(df[para_list[1]].tolist())

X = current
Y1 = voltage

# Calculate dV/dI
dV_dI = jnp.gradient(Y1, X)
Y2 = dV_dI

# Linear fit the dV/dI 
dV_dI_fit = constfit("dV_dI_fit", X, Y2)
print_fit_result(dV_dI_fit)

# Find the critical current Ic and retrapping current Ir using the two peaks of dV/dI curve
peaks_above_0, _ = find_peaks(dV_dI[X > 0], height=dV_dI_fit.intercept)
peaks_below_0, _ = find_peaks(dV_dI[X < 0], height=dV_dI_fit.intercept)
# Label the max peaks
max_peak_above_0 = peaks_above_0[jnp.argmax(dV_dI[X > 0][peaks_above_0])]
max_peak_below_0 = peaks_below_0[jnp.argmax(dV_dI[X < 0][peaks_below_0])]
# Assign Ic and Ir
Ic = X[X > 0][max_peak_above_0]
Ir = X[X < 0][max_peak_below_0]

# Linear fit the resistance R 
R_fit = polyfit("R_fit", X, Y1)
print_fit_result(R_fit)

# Linear fit the normal resistance Rn(in the range I >= Ic) and intercept Vn
Rn_fit = polyfit("Rn_fit", X[X >= Ic], Y1[X >= Ic])
print_fit_result(Rn_fit)


# Plot the RCSJ response for applied current from -2 µA to 2 µA using Plotly
fig = go.Figure()
# y2 Trace settings
_Y2 = dict(mode='lines', yaxis='y2')
# Add IV curve
fig.add_trace(go.Scatter(x=X, y=Y1, mode='lines', name='RCSJ IV Curve'))
# Add dV/dI curve with secondary y-axis
fig.add_trace(go.Scatter(x=X, y=Y2, name='<b>dV/dI</b>', line=dict(width=2), **_Y2))
# Add R linear fit
fig.add_trace(go.Scatter(x=X, y=X * R_fit.slope + R_fit.intercept, mode='lines', name=f'<b>R<sub>fit</sub>:</b> {SI.f(R_fit.slope, SI.Ω)}', line=dict(width=1, dash='dash')))
# Add Rn linear fit
fig.add_trace(go.Scatter(x=X[X >= Ic], y=X[X >= Ic] * Rn_fit.slope + Rn_fit.intercept, mode='lines', name=f'<b>Rn<sub>fit</sub>:</b> {SI.f(Rn_fit.slope, SI.Ω)}', line=dict(width=2, dash='dash')))
# Add dV/dI linear fit with secondary y-axis
fig.add_trace(go.Scatter(x=X, y=X * dV_dI_fit.slope + dV_dI_fit.intercept, name=f'<b>dV/dI<sub>fit</sub>:</b> {SI.f(dV_dI_fit.intercept, SI.Ω)}', line=dict(width=1,dash='dash'), **_Y2))
# Add vertical dashed line at x = Ic using go.Scatter
fig.add_trace(go.Scatter(x=[Ic, Ic], y=[min(Y2), max(Y2)], name=f'<b>Ic:</b> {SI.f(Ic, SI.A)}',line=dict(width=1, dash='dot'), **_Y2))
# Add vertical dashed line at x = Ir using go.Scatter
fig.add_trace(go.Scatter(x=[Ir, Ir], y=[min(Y2), max(Y2)], name=f'<b>Ir:</b> {SI.f(Ir, SI.A)}',line=dict(width=1, dash='dot'), **_Y2))
# Annotation settings
_N = dict(xref="paper", yref="paper", showarrow=False, font=dict(size=12,color="black"))
# Add annotations for IcRn
fig.add_annotation(x=0.5, y=1, text=f'<b>IcRn:</b> {SI.f(Ic * Rn_fit.slope, SI.V)}', **_N)
# Add annotations for Step size
fig.add_annotation(x=1, y=0, text=f'<b>Step size:</b> {SI.f(R_fit.step_size, SI.A)}', xanchor="right", **_N)

fig.update_layout(
    title=f'RCSJ IV Curve and dV/dI',
    xaxis_title='Applied Current (A)',
    yaxis=dict(
        title='Measured Voltage (V)',
    ),
    yaxis2=dict(
        title='dV/dI',
        overlaying='y',
        side='right'
    ),
    width=800,
    height=800,
    legend=dict(x=0, y=1),
    margin=dict(l=40, r=40, t=40, b=40),
    template='plotly_white'
)

fig.show()

# Save the plot as pdf file at current directory
fig.write_image("RCSJ_IV_curve.pdf",scale=2)
from pdf2image import convert_from_path
# Convert the pdf file to png file
images = convert_from_path("RCSJ_IV_curve.pdf")
for i, image in enumerate(images):
    image.save(f"RCSJ_IV_curve_{i}.png", "PNG", dpi=(1000, 1000))
# Delete the pdf file
import os
os.remove("RCSJ_IV_curve.pdf")



In [ ]:
from scipy import stats
from scipy.signal import find_peaks
import jax.numpy as jnp
import plotly.graph_objects as go

# Calculate dV/dI
dV_dI = jnp.gradient(voltages_measured, currents_applied)

# Find the critical current Ic and retrapping current Ir using the two peaks of dV/dI curve
peaks_above_0, _ = find_peaks(dV_dI[currents_applied > 0], height=0)
peaks_below_0, _ = find_peaks(dV_dI[currents_applied < 0], height=0)
Ic = currents_applied[peaks_above_0[0]]
Ir = currents_applied[peaks_below_0[0]]
print(f"Ic: {Ic:.4e} A")
print(f"Ir: {Ir:.4e} A")

# Linear fit the resistance R 
R_fit = polyfit("R_fit", currents_applied, voltages_measured)
print_fit_result(R_fit)

# Linear fit the normal resistance Rn(in the range I > Ic) and intercept Vn
Rn_fit = polyfit("Rn_fit", currents_applied[currents_applied > Ic], voltages_measured[currents_applied > Ic])
print_fit_result(Rn_fit)
print(Ic)
# Linear fit the dV/dI 
slope, dV_dI_fit = jnp.polyfit(currents_applied, dV_dI, 1)

dV_dI_mean = jnp.mean(dV_dI)

# Plot the RCSJ response for applied current from -2 µA to 2 µA using Plotly
fig = go.Figure()

# Add IV curve
fig.add_trace(go.Scatter(x=currents_applied, y=voltages_measured, mode='lines', name='RCSJ IV Curve'))
# Add R linear fit
fig.add_trace(go.Scatter(x=currents_applied, y=currents_applied * R_fit.slope + R_fit.intercept, mode='lines', name=f'<b>R<sub>fit</sub>:</b> {R_fit.slope:.4f}', line=dict(dash='dash')))
# Add Rn linear fit
fig.add_trace(go.Scatter(x=currents_applied[currents_applied > Ic], y=currents_applied[currents_applied > Ic] * Rn_fit.slope + Rn_fit.intercept, mode='lines', name=f'<b>Rn<sub>fit</sub>:</b> {Rn_fit.slope:.4f}', line=dict(dash='dash')))
# Add dV/dI curve with secondary y-axis
fig.add_trace(go.Scatter(x=currents_applied, y=dV_dI, mode='lines', name='<b>dV/dI</b>', line=dict(dash='dash'), yaxis='y2'))
# Add dV/dI linear fit with secondary y-axis
fig.add_trace(go.Scatter(x=currents_applied, y=currents_applied * slope + dV_dI_fit, mode='lines', name=f'<b>dV/dI<sub>fit</sub>:</b> {dV_dI_fit:.4f}', line=dict(dash='dash'), yaxis='y2'))

# Add peaks found
fig.add_trace(go.Scatter(x=currents_applied[currents_applied > 0][peaks_above_0], y=dV_dI[currents_applied > 0][peaks_above_0], mode='markers', name='Peaks Above 0', marker=dict(color='red', size=10)))
fig.add_trace(go.Scatter(x=currents_applied[currents_applied < 0][peaks_below_0], y=dV_dI[currents_applied < 0][peaks_below_0], mode='markers', name='Peaks Below 0', marker=dict(color='blue', size=10)))

# Label the max peaks
max_peak_above_0 = peaks_above_0[jnp.argmax(dV_dI[currents_applied > 0][peaks_above_0])]
max_peak_below_0 = peaks_below_0[jnp.argmax(dV_dI[currents_applied < 0][peaks_below_0])]

fig.add_trace(go.Scatter(
    x=[currents_applied[currents_applied > 0][max_peak_above_0]],
    y=[dV_dI[currents_applied > 0][max_peak_above_0]],
    mode='markers+text',
    name='Max Peak Above 0',
    text=[f'Max Peak Above 0: {dV_dI[currents_applied > 0][max_peak_above_0]:.4e}'],
    textposition='top center',
    marker=dict(color='red', size=12, symbol='x')
))

fig.add_trace(go.Scatter(
    x=[currents_applied[currents_applied < 0][max_peak_below_0]],
    y=[dV_dI[currents_applied < 0][max_peak_below_0]],
    mode='markers+text',
    name='Max Peak Below 0',
    text=[f'Max Peak Below 0: {dV_dI[currents_applied < 0][max_peak_below_0]:.4e}'],
    textposition='top center',
    marker=dict(color='blue', size=12, symbol='x')
))

fig.update_layout(
    title=f'RCSJ IV Curve and dV/dI<br>Ic={Ic} A, R={R} Ohm, C={C} F',
    xaxis_title='Applied Current (A)',
    yaxis=dict(
        title='Voltage (V)',
    ),
    yaxis2=dict(
        title='dV/dI',
        overlaying='y',
        side='right'
    ),
    legend=dict(x=0, y=1),
    margin=dict(l=40, r=40, t=40, b=40),
    template='plotly_white'
)

fig.show()

In [ ]:
def plot(currents_applied, voltages_measured):
    from scipy import stats
    # Calculate dV/dI
    dV_dI = jnp.gradient(voltages_measured, currents_applied)

    # Linear fit the resistance R 
    R_fit = polyfit("R_fit", currents_applied, voltages_measured)
    print_fit_result(R_fit)

    # Linear fit the normal resistance Rn(in the range I > Ic) and intercept Vn
    Rn_fit = polyfit("Rn_fit", currents_applied[currents_applied > Ic], voltages_measured[currents_applied > Ic])
    print_fit_result(Rn_fit)
    print(Ic)

    # Linear fit the dV/dI 
    dV_dI_fit = polyfit("dV_dI_fit", currents_applied, dV_dI)
    print_fit_result(dV_dI_fit)
    slope, dV_dI_fit = jnp.polyfit(currents_applied, dV_dI, 1)

    dV_dI_mean = jnp.mean(dV_dI)
    print(dV_dI_mean)

    # Plot the RCSJ response for applied current from -2 µA to 2 µA using Plotly
    fig = go.Figure()

    # Add IV curve
    fig.add_trace(go.Scatter(x=currents_applied, y=voltages_measured, mode='lines', name='RCSJ IV Curve'))
    # Add R linear fit
    fig.add_trace(go.Scatter(x=currents_applied, y=currents_applied * R_fit.slope + R_fit.intercept, mode='lines', name=f'R<sub>fit</sub>:{R_fit.slope:.4f}', line=dict(dash='dash')))
    # Add Rn linear fit
    fig.add_trace(go.Scatter(x=currents_applied[currents_applied > Ic], y=currents_applied[currents_applied > Ic] * Rn_fit.slope + Rn_fit.intercept, mode='lines', name=f'Rn<sub>fit</sub>:{Rn_fit.slope:.4f}', line=dict(dash='dash')))
    # Add dV/dI linear fit
    fig.add_trace(go.Scatter(x=currents_applied, y=currents_applied * slope + dV_dI_fit, mode='lines', name=f'{dV_dI_fit=:.4f}', line=dict(dash='dash'), yaxis='y2'))

    # Add dV/dI curve with secondary y-axis
    fig.add_trace(go.Scatter(x=currents_applied, y=dV_dI, mode='lines', name='dV/dI', line=dict(dash='dash'), yaxis='y2'))

    fig.update_layout(
        title=f'RCSJ IV Curve and dV/dI<br>Ic={Ic} A, R={R} Ohm, C={C} F',
        xaxis_title='Applied Current (A)',
        yaxis=dict(
            title='Voltage (V)',
        ),
        yaxis2=dict(
            title='dV/dI',
            overlaying='y',
            side='right'
        ),
        legend=dict(x=0, y=1),
        margin=dict(l=40, r=40, t=40, b=40),
        template='plotly_white'
    )

    fig.show()
plot(currents_applied, voltages_measured)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.stats import t

# Example data (x values and corresponding y values)
x = currents_applied[currents_applied > 3e-6]
y = dV_dI[currents_applied > 3e-6]

# Define the model for a constant function
def constant_model(x, c):
    return np.full_like(x, c)

# Use curve_fit to find the best-fit constant and covariance
params, covariance = curve_fit(constant_model, x, y)

# The fitted constant value (intercept)
fitted_constant = params[0]

# Residuals and R-squared calculation
y_fitted = constant_model(x, fitted_constant)
residuals = y - y_fitted
ss_res = np.sum(residuals**2)
ss_tot = np.sum((y - np.mean(y))**2)
r_squared = 1 - (ss_res / ss_tot)

# Standard deviation of the parameter estimate (for p-value)
stderr = np.sqrt(np.diag(covariance))

# Calculate the t-statistic and p-value for the constant parameter
t_stat = fitted_constant / stderr[0]
p_value = 2 * (1 - t.cdf(np.abs(t_stat), df=len(x)-1))

# Output results
print(f"Fitted constant value (intercept): {fitted_constant}")
print(f"Standard error of the fitted constant: {stderr[0]}")
print(f"t-statistic: {t_stat}")
print(f"p-value: {p_value}")
print(f"R-squared: {r_squared}")

# Plot the data and the fitted constant line
plt.scatter(x, y, color='blue', label='Data')
plt.plot(x, constant_model(x, fitted_constant), color='red', label=f"Fitted constant y = {fitted_constant}")
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.show()

In [ ]:
import jax.numpy as jnp
import matplotlib.pyplot as plt
from scipy.stats import t

x = currents_applied
y = dV_dI

# Use jnp.polyfit to find the best-fit constant
params = jnp.polyfit(x, y, 0)
fitted_constant = params[0]

# Residuals and R-squared calculation
y_fitted = jnp.full_like(x, fitted_constant)
residuals = y - y_fitted
ss_res = jnp.sum(residuals**2)
ss_tot = jnp.sum((y - jnp.mean(y))**2)
r_squared = 1 - (ss_res / ss_tot)

# Standard deviation of the parameter estimate (for p-value)
stderr = jnp.std(residuals) / jnp.sqrt(len(x))

# Calculate the t-statistic and p-value for the constant parameter
t_stat = fitted_constant / stderr
p_value = 2 * (1 - t.cdf(jnp.abs(t_stat), df=len(x)-1))
mean = jnp.mean(y)

# Output results
print(f"Fitted constant value (intercept): {fitted_constant}")
print(f"Mean: {mean}")
print(f"Mean-constant: {mean-fitted_constant}")
print(f"Standard error of the fitted constant: {stderr}")
print(f"t-statistic: {t_stat}")
print(f"p-value: {p_value}")
print(f"R-squared: {r_squared}")

# Plot the data and the fitted constant line
plt.scatter(x, y, color='blue', label='Data')
plt.plot(x, y_fitted, color='red', label=f"Fitted constant y = {fitted_constant}")
plt.xlabel('x')
plt.ylabel('y')
plt.legend()
plt.show()

In [ ]:
params = jnp.polyfit(x, y, 0)
params

In [ ]:
dV_dI_mean

In [ ]:
sol = diffrax.diffeqsolve(
    term,
    solver,
    t0=t[0],
    t1=t[-1],
    dt0=1e-6,
    y0=y0,
    args={"I_ext": I_ext},
    saveat=diffrax.SaveAt(ts=t),
    max_steps=100000  # Increased max_steps
)
